In [1]:
import json
import os
import folium
import numpy as np
from sklearn.neighbors import BallTree
from openai import OpenAI

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [34]:
"""
Homokhátság Water Management App
=================================
1. Loads farm and water body data from JSON
2. Calculates flood & drought scores from physical measurements
3. Uses KNN (BallTree + Haversine) to match each farm to its nearest water bodies
4. Generates LLM insights per farm via the Anthropic API
5. Renders an interactive Folium map with all data
"""


# ── Config ────────────────────────────────────────────────────────────────────

FARMS_FILE       = "farms.json"
WATER_FILE       = "water_bodies.json"
OUTPUT_MAP       = "water_map.html"
KNN_K            = 3        # how many nearest water bodies to find per farm
EARTH_RADIUS_KM  = 6371.0

# ── Reference thresholds for score calculation (per water body type) ──────────

# "Normal" operating water level (m) — what we'd expect in a balanced year
TYPE_NORMAL_LEVEL = {
    "river":      3.0,
    "canal":      1.5,
    "lake":       1.5,
    "reservoir":  2.5,
    "oxbow_lake": 2.0,
}

# Maximum plausible water level before overflow risk is near-certain
TYPE_MAX_LEVEL = {
    "river":      7.0,
    "canal":      2.5,
    "lake":       3.0,
    "reservoir":  4.0,
    "oxbow_lake": 4.5,
}

# "Normal" flow rate (m³/s) for flow-bearing body types
TYPE_NORMAL_FLOW = {
    "river":      500.0,
    "canal":       10.0,
    "lake":         0.0,
    "reservoir":    0.0,
    "oxbow_lake":   0.0,
}




In [35]:
# ── Score calculation ─────────────────────────────────────────────────────────

def clamp(value: float) -> float:
    """Clamp a float to [0.0, 1.0]."""
    return round(max(0.0, min(1.0, value)), 3)


def calculate_scores(wb: dict) -> tuple[float, float]:
    """
    Derive flood_score and drought_score from physical measurements.

    flood_score:
        Weighted combination of:
          - How far above the normal level the current level is,
            scaled to the available headroom before overflow  (weight 0.6)
          - How much the current flow exceeds the normal flow rate  (weight 0.4)
        Result: 0 = no flood risk, 1 = near-certain overflow

    drought_score:
        Weighted combination of:
          - How far below the normal level the current level is,
            as a fraction of the normal level  (weight 0.7)
          - How depleted the flow is relative to the normal flow  (weight 0.3)
        Result: 0 = no drought risk, 1 = critically dry
    """
    wtype  = wb["type"]
    level  = wb["water_level_m"]
    flow   = wb["avg_flow_m3s"] or 0.0

    normal_level = TYPE_NORMAL_LEVEL.get(wtype, 2.0)
    max_level    = TYPE_MAX_LEVEL.get(wtype, 5.0)
    normal_flow  = TYPE_NORMAL_FLOW.get(wtype, 0.0)

    # ── Flood score ────────────────────────────────────────────────────────
    headroom = max_level - normal_level
    level_flood = (level - normal_level) / headroom if headroom > 0 else 0.0

    if normal_flow > 0:
        # Scale flow against 3× the normal as the "danger" ceiling
        flow_flood = flow / (normal_flow * 3)
    else:
        flow_flood = 0.0

    flood_score = clamp(0.6 * level_flood + 0.4 * flow_flood)

    # ── Drought score ──────────────────────────────────────────────────────
    level_drought = (normal_level - level) / normal_level if normal_level > 0 else 0.0

    if normal_flow > 0 and flow > 0:
        flow_drought = clamp(1.0 - flow / normal_flow)
    elif normal_flow > 0 and flow == 0:
        flow_drought = 1.0          # no flow at all = maximum drought signal
    else:
        flow_drought = 0.0          # still-water bodies, flow irrelevant

    drought_score = clamp(0.7 * level_drought + 0.3 * flow_drought)

    return flood_score, drought_score



In [36]:

# ── KNN ───────────────────────────────────────────────────────────────────────

def build_knn_tree(water_bodies: list[dict]) -> BallTree:
    """Build a BallTree on water body coordinates (in radians for Haversine)."""
    coords = np.radians([[wb["lat"], wb["lon"]] for wb in water_bodies])
    return BallTree(coords, metric="haversine")


def find_nearest_water_bodies(
    farm: dict,
    water_bodies: list[dict],
    tree: BallTree,
    k: int = KNN_K,
) -> list[dict]:
    """
    Return the k nearest water bodies to a farm, each augmented with
    distance_km, flood_score, and drought_score.
    """
    farm_coord = np.radians([[farm["lat"], farm["lon"]]])
    distances, indices = tree.query(farm_coord, k=k)

    results = []
    for dist_rad, idx in zip(distances[0], indices[0]):
        wb = dict(water_bodies[idx])               # copy so we don't mutate source
        wb["distance_km"] = round(dist_rad * EARTH_RADIUS_KM, 2)
        results.append(wb)
    return results


# ── LLM insight ───────────────────────────────────────────────────────────────

def build_prompt(farm: dict, nearest: list[dict]) -> str:
    wb_lines = []
    for i, wb in enumerate(nearest, 1):
        wb_lines.append(
            f"  {i}. {wb['name']} ({wb['type']}) — "
            f"{wb['distance_km']} km away | "
            f"flood score: {wb['flood_score']} | "
            f"drought score: {wb['drought_score']} | "
            f"water level: {wb['water_level_m']} m | "
            f"notes: {wb['notes']}"
        )
    wb_block = "\n".join(wb_lines)

    return f"""You are a water management advisor for farmers in the Homokhátság region of Hungary, 
a semi-arid sandy area suffering from simultaneous drought and flood extremes.

Farm details:
- Name: {farm['name']}
- Owner: {farm['owner']}
- Crop: {farm['crop_type']}
- Size: {farm['size_ha']} ha
- Irrigation method: {farm['irrigation']}
- Annual water need: {farm['annual_water_need_m3']:,} m³

The {KNN_K} nearest water bodies are:
{wb_block}

Write a short, practical insight (3–5 sentences) for this farmer. 
Focus on:
- Whether any nearby water body has surplus water they could harness (high flood score = surplus available)
- Whether any nearby body is at drought risk and should not be relied on
- One concrete action the farmer can take to reduce reliance on government freshwater supply
- Be specific: mention the water body by name and distance

Tone: direct, friendly, actionable. No bullet points. Plain paragraph."""


def get_llm_insight(farm: dict, nearest: list[dict]) -> str:
    """Call the OpenAI API and return a plain-text insight for this farm."""


    token = os.environ["GITHUB_TOKEN"]
    endpoint = "https://models.github.ai/inference"
    model = "openai/gpt-4.1"

    client = OpenAI(
        base_url=endpoint,
        api_key=token,
    )

    prompt = build_prompt(farm, nearest)

    response = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "",
            },
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model=model
    )

    return response.choices[0].message.content.strip()




In [37]:
# ── Folium map ────────────────────────────────────────────────────────────────

def score_to_color(flood_score: float, drought_score: float) -> str:
    """
    Pick a marker color based on dominant risk:
      - High flood  → blue
      - High drought → orange
      - Both high   → red (conflicted / flash-flood-then-dry pattern)
      - Both low    → green (healthy)
    """
    if flood_score >= 0.5 and drought_score >= 0.4:
        return "red"
    elif flood_score >= 0.5:
        return "blue"
    elif drought_score >= 0.5:
        return "orange"
    else:
        return "green"


def wb_type_icon(wtype: str) -> str:
    icons = {
        "river":      "tint",
        "canal":      "arrows-h",
        "lake":       "circle",
        "reservoir":  "database",
        "oxbow_lake": "refresh",
    }
    return icons.get(wtype, "info-sign")


def build_map(
    farms: list[dict],
    water_bodies: list[dict],
    farm_matches: dict,        # farm_id → list of nearest wb dicts
    farm_insights: dict,       # farm_id → insight string
) -> folium.Map:

    center_lat = sum(f["lat"] for f in farms) / len(farms)
    center_lon = sum(f["lon"] for f in farms) / len(farms)
    m = folium.Map(location=[center_lat, center_lon], zoom_start=9, tiles="CartoDB positron")

    # ── Legend (HTML overlay) ──────────────────────────────────────────────
    legend_html = """
    <div style="position: fixed; bottom: 40px; left: 40px; z-index: 1000;
                background: white; padding: 14px 18px; border-radius: 8px;
                border: 1px solid #ccc; font-family: sans-serif; font-size: 13px;
                box-shadow: 2px 2px 6px rgba(0,0,0,0.15);">
      <b>Water body risk</b><br>
      <span style="color:#2196F3">&#9679;</span> Flood risk (level &gt; normal)<br>
      <span style="color:#FF9800">&#9679;</span> Drought risk (level &lt; normal)<br>
      <span style="color:#F44336">&#9679;</span> Both risks present<br>
      <span style="color:#4CAF50">&#9679;</span> Healthy / balanced<br>
      <hr style="margin:6px 0">
      <span style="color:#2E7D32">&#9632;</span> Farm<br>
      <span style="color:#777">&#9135;</span> Farm → nearest water body
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    # ── Water bodies ──────────────────────────────────────────────────────
    for wb in water_bodies:
        color = score_to_color(wb["flood_score"], wb["drought_score"])
        icon  = wb_type_icon(wb["type"])

        popup_html = f"""
        <div style="font-family:sans-serif;min-width:220px">
          <b style="font-size:14px">{wb['name']}</b><br>
          <span style="color:#666;font-size:12px">{wb['type'].replace('_',' ').title()}</span>
          <hr style="margin:6px 0">
          <table style="font-size:12px;width:100%">
            <tr><td>Water level</td><td><b>{wb['water_level_m']} m</b></td></tr>
            <tr><td>Flow rate</td>
                <td><b>{wb['avg_flow_m3s'] if wb['avg_flow_m3s'] else 'N/A'} m³/s</b></td></tr>
            <tr><td>Flood score</td>
                <td><b style="color:{'#2196F3' if wb['flood_score']>0.4 else '#333'}">
                    {wb['flood_score']}</b></td></tr>
            <tr><td>Drought score</td>
                <td><b style="color:{'#FF9800' if wb['drought_score']>0.4 else '#333'}">
                    {wb['drought_score']}</b></td></tr>
          </table>
          <hr style="margin:6px 0">
          <span style="font-size:11px;color:#555">{wb['notes']}</span>
        </div>
        """

        folium.Marker(
            location=[wb["lat"], wb["lon"]],
            icon=folium.Icon(color=color, icon=icon, prefix="fa"),
            popup=folium.Popup(popup_html, max_width=260),
            tooltip=f"{wb['name']} | flood: {wb['flood_score']} | drought: {wb['drought_score']}",
        ).add_to(m)

    # ── Farms + KNN lines + insights ──────────────────────────────────────
    for farm in farms:
        nearest = farm_matches.get(farm["id"], [])
        insight = farm_insights.get(farm["id"], "Insight not available.")

        # Build nearest-water summary for popup
        nearest_rows = "".join(
            f"<tr><td>{wb['name']}</td>"
            f"<td>{wb['distance_km']} km</td>"
            f"<td>F:{wb['flood_score']} D:{wb['drought_score']}</td></tr>"
            for wb in nearest
        )

        popup_html = f"""
        <div style="font-family:sans-serif;min-width:260px;max-width:300px">
          <b style="font-size:14px">{farm['name']}</b><br>
          <span style="color:#666;font-size:12px">{farm['owner']}</span>
          <hr style="margin:6px 0">
          <table style="font-size:12px;width:100%">
            <tr><td>Crop</td><td><b>{farm['crop_type'].title()}</b></td></tr>
            <tr><td>Size</td><td><b>{farm['size_ha']} ha</b></td></tr>
            <tr><td>Irrigation</td><td><b>{farm['irrigation']}</b></td></tr>
            <tr><td>Water need</td><td><b>{farm['annual_water_need_m3']:,} m³/yr</b></td></tr>
          </table>
          <hr style="margin:6px 0">
          <b style="font-size:12px">Nearest water bodies (KNN k={KNN_K})</b><br>
          <table style="font-size:11px;width:100%;border-collapse:collapse">
            <tr style="background:#f5f5f5">
              <th style="text-align:left">Name</th>
              <th>Dist.</th><th>Scores</th>
            </tr>
            {nearest_rows}
          </table>
          <hr style="margin:6px 0">
          <b style="font-size:12px">&#129668; Advisor insight</b><br>
          <p style="font-size:12px;color:#333;margin:4px 0">{insight}</p>
        </div>
        """

        folium.Marker(
            location=[farm["lat"], farm["lon"]],
            icon=folium.Icon(color="darkgreen", icon="leaf", prefix="fa"),
            popup=folium.Popup(popup_html, max_width=320),
            tooltip=f"{farm['name']} — click for insights",
        ).add_to(m)

        # Dashed line to the single nearest water body
        if nearest:
            closest = nearest[0]
            folium.PolyLine(
                locations=[
                    [farm["lat"], farm["lon"]],
                    [closest["lat"], closest["lon"]],
                ],
                color="#555",
                weight=1.5,
                dash_array="6 4",
                opacity=0.6,
                tooltip=f"{farm['name']} → {closest['name']} ({closest['distance_km']} km)",
            ).add_to(m)

    return m

In [38]:

# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    # 1. Load data
    print("Loading data...")
    with open(FARMS_FILE, encoding="utf-8") as f:
        farms = json.load(f)
    with open(WATER_FILE, encoding="utf-8") as f:
        water_bodies = json.load(f)

    # 2. Calculate scores
    print("Calculating flood & drought scores...")
    for wb in water_bodies:
        wb["flood_score"], wb["drought_score"] = calculate_scores(wb)
        print(
            f"  {wb['name'][:30]:30s}  "
            f"flood={wb['flood_score']:.3f}  drought={wb['drought_score']:.3f}"
        )

    # 3. Build KNN tree and match each farm
    print("\nRunning KNN (Haversine)...")
    tree = build_knn_tree(water_bodies)
    farm_matches = {}
    for farm in farms:
        farm_matches[farm["id"]] = find_nearest_water_bodies(farm, water_bodies, tree)
        nearest_name = farm_matches[farm["id"]][0]["name"]
        nearest_dist = farm_matches[farm["id"]][0]["distance_km"]
        print(f"  {farm['name'][:25]:25s} → {nearest_name} ({nearest_dist} km)")

    # 4. Generate LLM insights
    print("\nGenerating LLM insights...")
    api_key = os.environ.get("GITHUB_TOKEN")
    if not api_key:
        print("  GITHUB_TOKEN not set — skipping LLM insights.")
        farm_insights = {f["id"]: "LLM insight unavailable (API key not set)." for f in farms}
    else:
        farm_insights = {}
        for farm in farms:
            print(f"  Generating insight for {farm['name']}...")
            nearest = farm_matches[farm["id"]]
            farm_insights[farm["id"]] = get_llm_insight(farm, nearest)

    # 5. Build and save the Folium map
    print(f"\nBuilding map → {OUTPUT_MAP}")
    m = build_map(farms, water_bodies, farm_matches, farm_insights)
    m.save(OUTPUT_MAP)
    print(f"Done. Open {OUTPUT_MAP} in your browser.")



In [31]:
main()

Loading data...
Calculating flood & drought scores...
  Duna (Danube)                   flood=0.843  drought=0.000
  Tisza                           flood=0.611  drought=0.000
  Kiskunsági Főcsatorna           flood=0.000  drought=0.361
  Duna–Tisza-közi Hátság Canal    flood=0.000  drought=0.619
  Fehér-tó (Fehértó)              flood=0.000  drought=0.513
  Kondor-tó                       flood=0.000  drought=0.420
  Keleti Főcsatorna branch        flood=0.084  drought=0.111
  Ágasegyházi Tározó              flood=0.000  drought=0.056
  Kolon-tó                        flood=0.520  drought=0.000
  Alpári-Holt-Tisza               flood=0.264  drought=0.000
  Kiskőrösi Tározó                flood=0.000  drought=0.168
  Bócsai Belvízcsatorna           flood=0.341  drought=0.000

Running KNN (Haversine)...
  Kovács Tanya              → Kiskunsági Főcsatorna (3.82 km)
  Nagy-pusztai Gazdaság     → Kondor-tó (12.11 km)
  Homoki Birtok             → Duna–Tisza-közi Hátság Canal (4.63 km)
  Ki